## Prerequisties

Important: Only one kuberay worker pod should be created per k8s node

Minimum resources required for single worker 4 CPUs and 8 Gi memory 

Ensure minimal resource configuration is set, or update as below

Enable gpu as below

Update affinity for workergroup as below
```
affinity:
        podAntiAffinity:
          requiredDuringSchedulingIgnoredDuringExecution:
          - labelSelector:
              matchLabels:
                ray.io/group: smallGroup
            topologyKey: kubernetes.io/hostname

```

### For running xgboost train, update entrypoint to python `train_xgboost_model.py`

### For serving the trained xgboost model, update `entrypoint` to `""` and follow below workaround

Note: Following cell is a workaround. Currently, `serve deploy` does not support `--working-dir` directly.
Please see https://github.com/ray-project/ray/issues/29354

Suggested way to provide files from NB side to Ray cluster as below:
1. Create connection with `JobSubmissionClient` with working dir option but without entrypoint.
2. `JobSubmissionClient` will upload working_dir to GCS and print the URI.
3. Specify the above mentioned URI in config file as below example:
```
  runtime_env:
    working_dir: "gcs://_ray_pkg_fef565b457f470d9.zip"
```

In [9]:
import ray
import time
from ray.job_submission import JobSubmissionClient

ray_head_ip = "kuberay-head-svc.kuberay.svc.cluster.local"
ray_head_port = 8265
ray_address = f"http://{ray_head_ip}:{ray_head_port}"
client = JobSubmissionClient(ray_address)

In [10]:
!serve shutdown --address "http://kuberay-head-svc.kuberay.svc.cluster.local:8265" -y

2026-06-24 09:50:45,072	WARN scripts.py:766 -- No Serve instance found running on the cluster at http://kuberay-head-svc.kuberay.svc.cluster.local:8265.


In [4]:
!serve build --app-dir "./" serve_xgboost:xgboost_model -o xgboost_model_config.yaml

Module binding complete: `xgboost_model` is available for `serve build`.
2026-06-24 06:43:48,887	INFO scripts.py:886 -- The auto-generated application names default to `app1`, `app2`, ... etc. Rename as necessary.



In [ ]:
job_id = client.submit_job(
    entrypoint="python train_xgboost_model.py",
    runtime_env={
        "working_dir": "./",
        "pip": ["pandas==2.3.3","xgboost"],
        # Add proxy in env_vars if required
        "env_vars": {"RAY_TRAIN_V2_ENABLED": "1","HTTPS_PROXY": "http://hpeproxy.its.hpecorp.net:443","HTTP_PROXY": "http://hpeproxy.its.hpecorp.net:443", "https_proxy": "http://hpeproxy.its.hpecorp.net:443","http_proxy": "http://hpeproxy.its.hpecorp.net:443"}
    },
    entrypoint_num_gpus=1
)

print(f"Ray job submitted with job_id: {job_id}")

# Waiting for Ray to finish the job and print the result
while True:
    status = client.get_job_status(job_id)
    if status in [ray.job_submission.JobStatus.RUNNING, ray.job_submission.JobStatus.PENDING]:
        time.sleep(5)
    else:
        break
try:
    logs = client.get_job_logs(job_id)
    print(logs)
except RuntimeError as e:
    print(f"Failed to get job logs, please check logs on ray dashboard ")
# We do not need this connection    
ray.shutdown()

2026-06-24 09:50:48,321	INFO dashboard_sdk.py:356 -- Uploading package gcs://_ray_pkg_4ebbddbfd2230226.zip.
2026-06-24 09:50:48,322	INFO packaging.py:691 -- Creating a file package for local module './'.


Ray job submitted with job_id: raysubmit_dUSBPCnRSSkushj4


In [6]:
!serve deploy --address "http://kuberay-head-svc.kuberay.svc.cluster.local:8265" xgboost_model_config.yaml

2026-06-24 06:44:39,845	INFO scripts.py:244 -- Deploying from config file: 'xgboost_model_config.yaml'.
2026-06-24 06:44:41,735	SUCC scripts.py:364 -- 
Sent deploy request successfully.
 * Use `serve status` to check applications' statuses.
 * Use `serve config` to see the current application config(s).



In [7]:
import requests
sample_input = {
    "mean radius": 14.9,
    "mean texture": 22.53,
    "mean perimeter": 102.1,
    "mean area": 685.0,
    "mean smoothness": 0.09947,
    "mean compactness": 0.2225,
    "mean concavity": 0.2733,
    "mean concave points": 0.09711,
    "mean symmetry": 0.2041,
    "mean fractal dimension": 0.06898,
    "radius error": 0.253,
    "texture error": 0.8749,
    "perimeter error": 3.466,
    "area error": 24.19,
    "smoothness error": 0.006965,
    "compactness error": 0.06213,
    "concavity error": 0.07926,
    "concave points error": 0.02234,
    "symmetry error": 0.01499,
    "fractal dimension error": 0.005784,
    "worst radius": 16.35,
    "worst texture": 27.57,
    "worst perimeter": 125.4,
    "worst area": 832.7,
    "worst smoothness": 0.1419,
    "worst compactness": 0.709,
    "worst concavity": 0.9019,
    "worst concave points": 0.2475,
    "worst symmetry": 0.2866,
    "worst fractal dimension": 0.1155,
}
sample_target = 0  # Ground truth label

In [8]:
try:
    response = requests.post("http://kuberay-head-svc.kuberay.svc.cluster.local:8000/", json=sample_input)
    print('Response status:', response.status_code)
    print(response.json())
except requests.exceptions.RequestException as e:
    print(f"Request failed: {e}")

Response status: 200
0.032507430762052536


In [1]:
!serve shutdown --address "http://kuberay-head-svc.kuberay.svc.cluster.local:8265" -y

/bin/bash: line 1: serve: command not found
